# Chapter 12: Graphs

Model relationships, explore networks, find routes, and connect a network at minimum cost. The final example recommends learning topics using graph connections.

**Prerequisites:** JavaScript arrays, Map, Set, functions, classes, and basic recursion.

Select the **JavaScript (Node.js)** kernel and run all cells from top to bottom. No external packages are required. Every one of the guide’s 12 “Try it” tasks has a runnable worked solution with assertions. Restart the kernel before running the whole notebook again if it reports a repeated declaration.

## 1. Describe vertices and edges

A graph G = (V, E) contains vertices (entities) and edges (relationships). In a simple undirected graph, adjacent vertices share an edge; degree counts incident edges. A simple path has no repeated vertices. A cycle closes a path without repeating other vertices. A connected graph has a path between every pair.

The chapter’s graph has edges A–B, A–C, A–D, C–D, C–G, D–G, D–H, B–E, B–F, E–I. Thus A has degree 3, A–C–D–A is a cycle, and A–B–E–I is a three-edge path.

In [ ]:
const chapterEdges = [
    ['A', 'B'], ['A', 'C'], ['A', 'D'], ['C', 'D'], ['C', 'G'],
    ['D', 'G'], ['D', 'H'], ['B', 'E'], ['B', 'F'], ['E', 'I']
];

### Try it

List the neighbors of D, its degree, and one simple path from A to H.

Predict the answer before running the worked solution below.

**Worked solution 1.** D has neighbors A, C, G, H and degree 4. A–D–H is a simple path.

In [ ]:
(() => {
    const assert = require("node:assert/strict");
const neighbors = chapterEdges.flatMap(([u, v]) => u === 'D' ? [v] : v === 'D' ? [u] : []);
assert.deepEqual(neighbors, ['A', 'C', 'G', 'H']);
const path = ['A', 'D', 'H'];
assert.equal(new Set(path).size, path.length);
assert.ok(path.slice(1).every((v, i) => chapterEdges.some(([a, b]) =>
    (a === path[i] && b === v) || (b === path[i] && a === v))));
console.log({ neighbors, degree: neighbors.length, path });
})();

## 2. Choose a graph representation

Undirected edges work both ways; directed edges permit movement in one direction. Weights represent costs or scores. Indegree counts incoming directed edges, and outdegree counts outgoing edges. Strong connectivity means every vertex can reach every other vertex in a directed graph. Sparsity concerns the number of edges, a separate property.

| Representation | Storage | Edge lookup |
| --- | --- | --- |
| Adjacency list of arrays | O(V + E) | O(degree of u) |
| Adjacency matrix | O(V²) | O(1) with indexed vertices |
| Dense incidence matrix | O(VE) | Depends on operation |

An incidence matrix has a column per edge and a row per vertex. Adjacency lists suit sparse networks. Dense matrices reserve space for missing connections. Weighted matrices here use 0 on the diagonal and Infinity for a missing edge; zero-weight edges are valid.

In [ ]:
const triangleList = { A: ['B', 'C'], B: ['A', 'C'], C: ['A', 'B'] };
const triangleAdjacency = [[0, 1, 1], [1, 0, 1], [1, 1, 0]];
// Rows A, B, C; columns AB, AC, BC.
const triangleIncidence = [[1, 1, 0], [1, 0, 1], [0, 1, 1]];
console.log(triangleList);
console.table(triangleAdjacency);
console.table(triangleIncidence);

### Try it 

Which representation suits 10,000 vertices with only a few neighbors each?

Predict the answer before running the worked solution below.

**Worked solution 2.** Use an adjacency list: O(V + E) storage instead of O(V²) dense matrix entries.

In [ ]:
(() => {
    const assert = require("node:assert/strict");
const vertices = 10_000;
const averageDegree = 4;
const neighborEntries = vertices * averageDegree;
assert.equal(vertices ** 2, 100_000_000);
assert.equal(neighborEntries, 40_000);
console.log({ matrixEntries: vertices ** 2, adjacencyListNeighborEntries: neighborEntries });
console.log('The list also stores the 10,000 vertices; these are entry counts, not byte estimates.');
})();

## 3. Build a graph with Map and Set

Map stores vertices and Set stores unique neighbors. An undirected edge appears in both neighbor sets. This class stores unweighted connections without parallel edges. The examples use distinct endpoints (no self-loops). Map and Set iterate in insertion order, which determines traversal tie-breaking.

In [ ]:
class Graph {
   constructor(directed = false) {
     this.directed = directed;
     this.adj = new Map();
   }
   addVertex(v) {
     if (!this.adj.has(v)) this.adj.set(v, new Set());
   }
   addEdge(u, v) {
     this.addVertex(u);
     this.addVertex(v);
     this.adj.get(u).add(v);
     if (!this.directed) this.adj.get(v).add(u);
   }
   vertices() { return [...this.adj.keys()]; }
   neighbors(v) { return this.adj.get(v) ?? new Set(); }
 }

const graph = new Graph();
 const edges = [
   ["A", "B"], ["A", "C"], ["A", "D"],
   ["C", "D"], ["C", "G"], ["D", "G"],
   ["D", "H"], ["B", "E"], ["B", "F"], ["E", "I"]
 ];
 edges.forEach(([u, v]) => graph.addEdge(u, v));
 console.log([...graph.neighbors("A")]);
 // ["B", "C", "D"]

### Try it 

Add isolated J, then repeat A–B. Does A’s degree change?

Predict the answer before running the worked solution below.

**Worked solution 3.** J has an empty neighbor set. A’s degree stays 3 because Set stores B only once. This solution adds J to the shared graph for later lessons.

In [ ]:
(() => {
    const assert = require("node:assert/strict");
graph.addVertex('J');
const before = graph.neighbors('A').size;
graph.addEdge('A', 'B');
assert.equal(graph.neighbors('J').size, 0);
assert.equal(graph.neighbors('A').size, before);
assert.equal(before, 3);
console.log({ vertices: graph.vertices(), degreeA: before, neighborsJ: [...graph.neighbors('J')] });
})();

## 4. Explore in layers with BFS

Breadth-first search uses a queue and marks vertices when enqueued, preventing repeated discovery through cycles. A head index avoids moving elements with shift. With adjacency lists, traversal takes O(V + E) over the reachable subgraph and O(V) auxiliary space. Unreachable vertices are absent from distance.

In [ ]:
function bfs(graph, start) {
   if (!graph.adj.has(start)) throw new Error("Unknown start");
   const queue = [start];
   const distance = new Map([[start, 0]]);
   const parent = new Map([[start, null]]);
   for (let head = 0; head < queue.length; head++) {
     const u = queue[head];
     for (const v of graph.neighbors(u)) {
       if (distance.has(v)) continue;
       distance.set(v, distance.get(u) + 1);
       parent.set(v, u);
       queue.push(v);
     }
   }
   return { order: queue, distance, parent };
 }
 const result = bfs(graph, "A");
 console.log(result.order.join(" "));
 // A B C D E F G H I

### Try it 

What is the distance from A to I? Does BFS from A include isolated J?

Predict the answer before running the worked solution below.

**Worked solution 4.** I is three edges from A. J is not reached.

In [ ]:
(() => {
    const assert = require("node:assert/strict");
const result = bfs(graph, 'A');
assert.equal(result.distance.get('I'), 3);
assert.equal(result.distance.has('J'), false);
assert.equal(result.order.includes('J'), false);
console.log({ distanceToI: result.distance.get('I'), reachesJ: result.distance.has('J') });
})();

## 5. Reconstruct an unweighted shortest path

Follow parent links backward from a reachable target, then reverse the sequence. BFS minimizes the number of edges, also minimizing total cost when each edge has the same positive cost. It does not minimize unequal weights.

From A: B/C/D have distance 1; E/F/G/H have distance 2; I has distance 3. G is first discovered through C. J is unreachable. These examples use string vertex IDs; null is reserved as the root-parent sentinel.

In [ ]:
function pathTo(result, target) {
   if (!result.distance.has(target)) return [];
   const path = [];
   for (let v = target; v !== null; v = result.parent.get(v)) {
     path.push(v);
   }
   return path.reverse();
 }
 const fromA = bfs(graph, "A");
 console.log(pathTo(fromA, "I")); // ["A", "B", "E", "I"]
 console.log(pathTo(fromA, "J")); // [] if J is unreachable

### Try it

Find the returned path for H. Why check reachability before following parents?

Predict the answer before running the worked solution below.

**Worked solution 5.** The path is A–D–H. An unreachable target has no chain to A; return [] before looking up missing parent links.

In [ ]:
(() => {
    const assert = require("node:assert/strict");
const result = bfs(graph, 'A');
assert.deepEqual(pathTo(result, 'H'), ['A', 'D', 'H']);
assert.deepEqual(pathTo(result, 'J'), []);
assert.deepEqual(pathTo(result, 'A'), ['A']);
console.log('H:', pathTo(result, 'H'), 'J:', pathTo(result, 'J'));
})();

## 6. Explore deeply with DFS

DFS follows an unvisited neighbor until it must backtrack. The outer loop visits every component, including isolated vertices. Discovery and finish times share one counter: each vertex has discovery < finish, and the complete traversal assigns 2V timestamps.

With adjacency lists, DFS takes O(V + E) time and O(V) auxiliary space. Deep recursion can exceed JavaScript’s call stack; use an explicit stack for large graphs.

In [ ]:
function dfs(graph) {
   const seen = new Set();
   const order = [];
   const discovery = new Map();
   const finish = new Map();
   let time = 0;
   function visit(u) {
     seen.add(u);
     order.push(u);
     discovery.set(u, ++time);
     for (const v of graph.neighbors(u)) {
       if (!seen.has(v)) visit(v);
     }
     finish.set(u, ++time);
   }
   for (const u of graph.vertices()) {
     if (!seen.has(u)) visit(u);
   }
   return { order, discovery, finish };
 }
 console.log(dfs(graph).order.join(" "));
 // A B E I F C D G H J (J was added in Try it 3)

### Try it 

Why does DFS not guarantee the fewest edges? What does the outer loop add for isolated J?

Predict the answer before running the worked solution below.

**Worked solution 6.** DFS commits to a branch before exploring alternatives. The outer loop visits J as a separate DFS tree. In the counterexample below DFS reaches H through B and C before seeing the direct edge A–H.

In [ ]:
(() => {
    const assert = require("node:assert/strict");
const counterexample = new Graph();
[['A', 'B'], ['B', 'C'], ['C', 'H'], ['A', 'H']].forEach(([u, v]) => counterexample.addEdge(u, v));
assert.deepEqual(dfs(counterexample).order, ['A', 'B', 'C', 'H']);
assert.deepEqual(pathTo(bfs(counterexample, 'A'), 'H'), ['A', 'H']);
const result = dfs(graph);
assert.ok(result.order.includes('J'));
assert.equal(result.order.length, graph.vertices().length);
assert.ok(result.discovery.get('J') < result.finish.get('J'));
console.log('DFS counterexample order:', dfs(counterexample).order);
console.log('Full chapter traversal, including J:', result.order);
})();

## 7. Order dependencies with a DAG

A directed acyclic graph has no directed cycle. If u → v means u must happen before v, every topological order places u before v. DFS appends finished vertices and reverses them. An edge to an active vertex reveals a directed cycle. Time is O(V + E), with O(V) extra space; recursion has the same depth limit as DFS.

In [ ]:
function topologicalSort(graph) {
   if (!graph.directed) throw new Error("Use a directed graph");
   const state = new Map(); // absent: unseen, 1: active, 2: done
   const order = [];
   function visit(u) {
     if (state.get(u) === 1) throw new Error("Directed cycle");
     if (state.get(u) === 2) return;
     state.set(u, 1);
     for (const v of graph.neighbors(u)) visit(v);
     state.set(u, 2);
     order.push(u);
   }
   for (const u of graph.vertices()) visit(u);
   return order.reverse();
 }

In [ ]:
const dependencyGraph = new Graph(true);
const dependencyEdges = [['A', 'C'], ['A', 'D'], ['B', 'D'], ['B', 'E'], ['C', 'F'], ['F', 'E']];
dependencyEdges.forEach(([u, v]) => dependencyGraph.addEdge(u, v));
console.log('One valid order:', topologicalSort(dependencyGraph));

### Try it 

Is A, B, C, D, F, E valid? What happens after adding E → A?

Predict the answer before running the worked solution below.

**Worked solution 7.** Yes, every prerequisite precedes its dependent. E → A creates the cycle A → C → F → E → A, so sorting throws. Use a separate graph to preserve the original DAG.

In [ ]:
(() => {
    const assert = require("node:assert/strict");
const proposed = ['A', 'B', 'C', 'D', 'F', 'E'];
assert.ok(dependencyEdges.every(([u, v]) => proposed.indexOf(u) < proposed.indexOf(v)));
const cyclic = new Graph(true);
dependencyEdges.forEach(([u, v]) => cyclic.addEdge(u, v));
cyclic.addEdge('E', 'A');
assert.throws(() => topologicalSort(cyclic), /Directed cycle/);
console.log('Proposed order is valid. Adding E → A is rejected as a directed cycle.');
})();

## 8. Find weighted routes with Dijkstra

Dijkstra settles the nearest tentative vertex, then relaxes its outgoing edges. It requires **nonnegative weights**. The matrix implementation takes O(V²) time and O(V) auxiliary space. A heap with adjacency lists improves performance on sparse graphs.

Use a square matrix, valid source index, finite numeric weights or Infinity, and zero diagonal. A–B costs 2, B–C costs 1, and direct A–C costs 4. The matrix is defined here so this lesson runs before Floyd–Warshall.

In [ ]:
const I = Infinity;
 const matrix = [
   [0, 2, 4, I, I, I],
   [I, 0, 1, 4, 2, I],
   [I, I, 0, I, 3, I],
   [I, I, I, 0, I, 2],
   [I, I, I, 3, 0, 2],
   [I, I, I, I, I, 0]
 ];

In [ ]:
function dijkstra(matrix, start) {
   const n = matrix.length;
   const distance = Array(n).fill(Infinity);
   const parent = Array(n).fill(null);
   const done = Array(n).fill(false);
   distance[start] = 0;
   for (let step = 0; step < n; step++) {
     let u = -1;
     for (let v = 0; v < n; v++)
       if (!done[v] && (u === -1 || distance[v] < distance[u]))
         u = v;
     if (u === -1 || distance[u] === Infinity) break;
     done[u] = true;
     for (let v = 0; v < n; v++) {
       const w = matrix[u][v];
       if (!done[v] && w !== Infinity &&
           distance[u] + w < distance[v]) {
         distance[v] = distance[u] + w;
         parent[v] = u;
       }
     }
   }
   return { distance, parent };
 }

In [ ]:
console.log('Distances from A:', dijkstra(matrix, 0).distance); // [0, 2, 3, 6, 4, 6]

### Try it 

Why is the distance to C 3? Why are negative edges unsafe?

Predict the answer before running the worked solution below.

**Worked solution 8.** A → B → C costs 2 + 1 = 3. A negative edge can improve an already settled vertex, invalidating the greedy choice. The negative-edge example intentionally violates Dijkstra’s precondition.

In [ ]:
(() => {
    const assert = require("node:assert/strict");
assert.equal(dijkstra(matrix, 0).distance[2], 3);
const unsafe = [[0, 2, 5], [Infinity, 0, Infinity], [Infinity, -10, 0]];
assert.equal(dijkstra(unsafe, 0).distance[1], 2);
console.log('Negative-edge counterexample: Dijkstra settles B at 2, but A → C → B costs', 5 - 10);
console.log('Use an algorithm that supports negative weights for such inputs.');
})();

## 9. Find routes between every pair

Floyd–Warshall asks whether going from i to j through k improves the distance. The intermediate vertex loop must be outermost. It takes O(V³) time and O(V²) space for a copied distance matrix.

Negative edges are allowed. A negative final diagonal reveals a negative cycle; routes that can pass through such a cycle have no finite minimum. Infinity denotes no route. Distances alone do not reconstruct paths.

In [ ]:
function floydWarshall(matrix) {
   const distance = matrix.map(row => [...row]);
   const n = distance.length;
   for (let k = 0; k < n; k++) {
     for (let i = 0; i < n; i++) {
       for (let j = 0; j < n; j++) {
         distance[i][j] = Math.min(
           distance[i][j],
           distance[i][k] + distance[k][j]
         );
       }
     }
   }
   return distance;
 }

In [ ]:
console.log('All-pairs distances:');
console.table(floydWarshall(matrix));

### Try it 

Why does the F-to-A distance stay Infinity? What does a negative final diagonal mean?

Predict the answer before running the worked solution below.

**Worked solution 9.** F has no outgoing route to A. A negative diagonal means a negative cycle. Inspect the computed distance matrix, since the function does not mutate its input.

In [ ]:
(() => {
    const assert = require("node:assert/strict");
const distances = floydWarshall(matrix);
assert.equal(distances[5][0], Infinity);
const negativeCycle = floydWarshall([[0, 1], [-2, 0]]);
assert.ok(negativeCycle.some((row, i) => row[i] < 0));
console.log('F → A:', distances[5][0], 'Negative-cycle diagonal:', negativeCycle.map((row, i) => row[i]));
})();

## 10. Connect a network with Prim

A minimum spanning tree connects a connected undirected weighted graph with V − 1 edges, no cycles, and minimum total weight. It minimizes the whole network cost, rather than each route from a source.

Prim’s key is the cheapest edge joining an outside vertex to the current tree, unlike Dijkstra’s accumulated path cost. Use a symmetric matrix with Infinity for missing edges and 0 on the diagonal. Negative and zero weights are valid. This version takes O(V²) time and O(V) extra space; disconnected inputs throw.

In [ ]:
function prim(matrix) {
   const n = matrix.length;
   if (n === 0) return [];
   const key = Array(n).fill(Infinity);
   const parent = Array(n).fill(null);
   const used = Array(n).fill(false);
   key[0] = 0;
   for (let step = 0; step < n; step++) {
     let u = -1;
     for (let v = 0; v < n; v++)
       if (!used[v] && (u === -1 || key[v] < key[u])) u = v;
     if (u === -1 || key[u] === Infinity)
       throw new Error("Graph is disconnected");
     used[u] = true;
     for (let v = 0; v < n; v++) {
       if (!used[v] && matrix[u][v] < key[v]) {
         key[v] = matrix[u][v];
         parent[v] = u;
       }
     }
   }
   return parent;
 }

In [ ]:
const networkVertices = ['A', 'B', 'C', 'D', 'E', 'F'];
const networkEdges = [
    ['A', 'B', 2], ['A', 'C', 4], ['B', 'C', 2],
    ['B', 'D', 4], ['B', 'E', 2], ['C', 'E', 3],
    ['D', 'E', 3], ['D', 'F', 2], ['E', 'F', 2]
];
const networkMatrix = networkVertices.map((_, i) => networkVertices.map((_, j) => i === j ? 0 : Infinity));
networkEdges.forEach(([u, v, weight]) => {
    const a = networkVertices.indexOf(u), b = networkVertices.indexOf(v);
    networkMatrix[a][b] = networkMatrix[b][a] = weight;
});
const primParents = prim(networkMatrix);
const primEdges = primParents.flatMap((parent, v) => parent === null ? [] :
    [[networkVertices[parent], networkVertices[v], networkMatrix[parent][v]]]);
console.log('Prim edges:', primEdges);
console.log('Total weight:', primEdges.reduce((sum, edge) => sum + edge[2], 0));

### Try it 

How many edges does a spanning tree on six vertices contain? Can an MST contain a cycle?

Predict the answer before running the worked solution below.

**Worked solution 10.** Five edges. A tree is connected and acyclic; a cycle would allow an edge to be removed while preserving connectivity.

In [ ]:
(() => {
    const assert = require("node:assert/strict");
assert.equal(primEdges.length, 6 - 1);
assert.equal(primEdges.reduce((sum, edge) => sum + edge[2], 0), 10);
assert.throws(() => prim([[0, Infinity], [Infinity, 0]]), /disconnected/);
console.log('Six vertices require five tree edges, with no cycles. MST weight:', 10);
})();

## 11. Connect components with Kruskal

Kruskal sorts edges by weight, accepting an edge only when its endpoints belong to different components. A disjoint set uses path compression and union by size. Inputs must list every endpoint in vertices and include each undirected edge once. Disconnected inputs return a minimum spanning forest.

The implementation copies the edge list before sorting. The general bound is O(V + E log E + E α(V)), where α is the inverse Ackermann function; sorting dominates typical connected simple graphs. Auxiliary space is O(V + E).

In [ ]:
function kruskal(vertices, edges) {
   const parent = new Map(vertices.map(v => [v, v]));
   const size = new Map(vertices.map(v => [v, 1]));
   function find(v) {
     if (parent.get(v) !== v) parent.set(v, find(parent.get(v)));
     return parent.get(v);
   }
   const selected = [];
   for (const [u, v, weight] of [...edges].sort((a,b) => a[2]-b[2])) {
     let a = find(u), b = find(v);
     if (a === b) continue; // This edge would create a cycle.
     if (size.get(a) < size.get(b)) [a, b] = [b, a];
     parent.set(b, a);
     size.set(a, size.get(a) + size.get(b));
     selected.push([u, v, weight]);
     if (selected.length === vertices.length - 1) break;
   }
   return selected;
 }

In [ ]:
const mstEdges = kruskal(networkVertices, networkEdges);
console.log('Kruskal edges:', mstEdges);
console.log('Total weight:', mstEdges.reduce((sum, edge) => sum + edge[2], 0));

### Try it 

Why skip A–C if A, B, and C already belong to one component?

Predict the answer before running the worked solution below.

**Worked solution 11.** A and C already have a connecting path. Adding another edge between them creates a cycle.

In [ ]:
(() => {
    const assert = require("node:assert/strict");
const selected = kruskal(['A', 'B', 'C'], [['A', 'B', 1], ['B', 'C', 2], ['A', 'C', 3]]);
assert.deepEqual(selected, [['A', 'B', 1], ['B', 'C', 2]]);
console.log('Accepted:', selected, 'Skipped A–C: endpoints already connected.');
})();

## 12. Use a graph for AI recommendations

Model learners and topics as a bipartite undirected graph: every edge joins a user: ID to a topic: ID. Recommend topics studied by peers who share known topics.

Scores count three-edge paths user → shared topic → peer → new topic. One peer can contribute more than once through different shared topics. This is a transparent heuristic, with no model training. A learned ranker could later combine it with content similarity or feedback. Runtime depends on the number of these paths examined, followed by sorting candidate scores.

In [ ]:
function recommendTopics(graph, user) {
   const known = graph.neighbors(user);
   const scores = new Map();
   for (const topic of known) {
     for (const peer of graph.neighbors(topic)) {
       if (peer === user) continue;
       for (const candidate of graph.neighbors(peer)) {
         if (known.has(candidate)) continue;
         scores.set(candidate, (scores.get(candidate) ?? 0) + 1);
       }
     }
   }
   return [...scores].sort((a, b) => b[1] - a[1]);
 }
 const learning = new Graph();
 [
   ["user:Ada", "topic:Search"],
   ["user:Ada", "topic:Learning"],
   ["user:Ben", "topic:Search"],
   ["user:Ben", "topic:Graphs"],
   ["user:Cara", "topic:Graphs"]
 ].forEach(([u, v]) => learning.addEdge(u, v));
 console.log(recommendTopics(learning, "user:Ada"));
 // [["topic:Graphs", 1]]

### Try it 

Add Ben–Planning and Cara–Search. Predict Ada’s recommendations and scores. Why might popular topics be over-recommended?

Predict the answer before running the worked solution below.

**Worked solution .** Graphs scores 2, through Ben and Cara; Planning scores 1, through Ben. Popular topics accumulate many paths even when they are a weak personal match.

In [ ]:
(() => {
    const assert = require("node:assert/strict");
learning.addEdge('user:Ben', 'topic:Planning');
learning.addEdge('user:Cara', 'topic:Search');
const recommendations = recommendTopics(learning, 'user:Ada');
assert.deepEqual(recommendations, [['topic:Graphs', 2], ['topic:Planning', 1]]);
console.log(recommendations);
console.log('Popularity can inflate path counts; relevance needs additional signals.');
})();

## 13. Apply the ideas: a study plan

Build a directed prerequisite graph, reject cycles, and display a valid study order. Keep the learner/topic recommendation graph separate from prerequisite relationships.

In [ ]:
const studyPlan = new Graph(true);
[['Arrays', 'Graphs'], ['Map and Set', 'Graphs'], ['Graphs', 'BFS'],
 ['Graphs', 'DFS'], ['DFS', 'Topological sort'], ['BFS', 'Shortest paths']]
    .forEach(([prerequisite, topic]) => studyPlan.addEdge(prerequisite, topic));
console.log('Study order:', topologicalSort(studyPlan));
console.log('Ada could explore:', recommendTopics(learning, 'user:Ada'));

## 14. Checks and review

The final checks cover traversal, disconnected graphs, cycle rejection, zero-weight edges, negative edges where supported, MST agreement, and recommendation scores. The 12 worked solutions above also assert their own answers.

In [ ]:
(() => {
    const assert = require('node:assert/strict');
    assert.deepEqual(bfs(graph, 'A').order, ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I']);
    assert.throws(() => bfs(graph, 'unknown'), /Unknown start/);
    assert.deepEqual(pathTo(bfs(graph, 'J'), 'J'), ['J']);
    assert.deepEqual(pathTo(bfs(graph, 'J'), 'A'), []);
    const traversal = dfs(graph);
    assert.deepEqual(traversal.order, ['A', 'B', 'E', 'I', 'F', 'C', 'D', 'G', 'H', 'J']);
    const times = graph.vertices().flatMap(v => {
        assert.ok(traversal.discovery.get(v) < traversal.finish.get(v));
        return [traversal.discovery.get(v), traversal.finish.get(v)];
    });
    assert.deepEqual(times.sort((a, b) => a - b), Array.from({ length: 20 }, (_, i) => i + 1));
    const order = topologicalSort(dependencyGraph);
    assert.ok(dependencyEdges.every(([u, v]) => order.indexOf(u) < order.indexOf(v)));
    assert.throws(() => topologicalSort(graph), /directed graph/);
    const selfCycle = new Graph(true);
    selfCycle.addEdge('A', 'A');
    assert.throws(() => topologicalSort(selfCycle), /Directed cycle/);
    const original = matrix.map(row => row.slice());
    const allPairs = floydWarshall(matrix);
    for (let start = 0; start < matrix.length; start++) {
        assert.deepEqual(dijkstra(matrix, start).distance, allPairs[start]);
    }
    assert.deepEqual(allPairs[0], [0, 2, 3, 6, 4, 6]);
    assert.deepEqual(matrix, original);
    const zeroWeight = [[0, 0, 5], [Infinity, 0, 2], [Infinity, Infinity, 0]];
    assert.deepEqual(dijkstra(zeroWeight, 0).distance, [0, 0, 2]);
    assert.deepEqual(floydWarshall([[0, -1], [Infinity, 0]])[0], [0, -1]);
    assert.deepEqual(prim([]), []);
    assert.deepEqual(prim([[0]]), [null]);
    assert.deepEqual(prim([[0, -2], [-2, 0]]), [null, 0]);
    assert.equal(mstEdges.length, 5);
    assert.equal(mstEdges.reduce((sum, edge) => sum + edge[2], 0), 10);
    assert.equal(primEdges.reduce((sum, edge) => sum + edge[2], 0), 10);
    assert.deepEqual(kruskal(['A', 'B', 'C'], [['A', 'B', 0]]), [['A', 'B', 0]]);
    assert.deepEqual(kruskal([], []), []);
    assert.deepEqual(recommendTopics(learning, 'user:unknown'), []);
    assert.deepEqual(recommendTopics(learning, 'user:Ada'), [['topic:Graphs', 2], ['topic:Planning', 1]]);
    console.log('All Chapter 12 checks passed.');
})();

## Choose your next algorithm

| Goal | Algorithm | Time for shown implementation |
| --- | --- | --- |
| Fewest links | BFS | O(V + E) |
| Structural exploration | DFS | O(V + E) |
| Prerequisite order in a DAG | Topological sort | O(V + E) |
| One-source nonnegative route costs | Dijkstra | O(V²) |
| All-pairs route costs | Floyd–Warshall | O(V³) |
| Minimum total connection cost | Prim / Kruskal | O(V²) / sorting plus disjoint-set operations |

Adapted from the supplied Chapter 12 Graphs guide. All 12 lesson tasks have worked solutions above. Examples preserve insertion order so the traversal outputs are predictable.